In [1]:
from vllm import LLM, SamplingParams
from transformers import AutoTokenizer, GemmaTokenizerFast

import pandas as pd
import numpy as np
from tqdm import tqdm
import os

import torch
import json
import re

import faiss
from langchain_community.vectorstores import FAISS
from FlagEmbedding import FlagReranker
from langchain_huggingface.embeddings import HuggingFaceEmbeddings

ROOT_DIR = "/project/lt200304-dipmt/paweekorn"
RETRIEVER= "bge-m3"
MODEL_PATH = "./models/openthaigpt-8b"

print("GPU count:", torch.cuda.device_count())

[2026-03-29 06:42:46] INFO _loader.py:132: Loading faiss with AVX2 support.
[2026-03-29 06:42:46] INFO _loader.py:137: Could not load library with AVX2 support due to:
ModuleNotFoundError("No module named 'faiss.swigfaiss_avx2'")
[2026-03-29 06:42:46] INFO _loader.py:156: Loading faiss.
[2026-03-29 06:42:46] INFO _loader.py:158: Successfully loaded faiss.
[2026-03-29 06:42:46] INFO config.py:54: PyTorch version 2.10.0 available.


GPU count: 1


## Overview

In [2]:
test_df = pd.read_csv("./data/questions.csv")
print(test_df.shape)
test_df.head()

(100, 12)


,id,question,choice_1,choice_2,choice_3,choice_4,choice_5,choice_6,choice_7,choice_8,choice_9,choice_10
0,1,Watch S3 Ultra กันน้ำได้กี่ ATM ครับ,3 ATM,IP68,5 ATM,IP67,10 ATM,20 ATM,IPX8,1 ATM,ไม่มีข้อมูลนี้ในฐานข้อมูล,คำถามนี้ไม่เกี่ยวข้องกับฐานข้อมูลสินค้า
1,2,ปากกา SaiFah Pen Gen 2 ราคาเท่าไหร่คะ,"2,990 บาท","4,490 บาท","1,990 บาท","4,990 บาท","3,490 บาท","2,490 บาท","3,990 บาท","5,490 บาท",ไม่มีข้อมูลนี้ในฐานข้อมูล,คำถามนี้ไม่เกี่ยวข้องกับฐานข้อมูลสินค้า
2,3,หูฟัง HeadPro X1 ใช้ Bluetooth เวอร์ชันอะไรคะ,Bluetooth 5.0,Bluetooth 5.3,Bluetooth 5.4,Bluetooth 4.2,Bluetooth 5.2,Bluetooth 5.1,Bluetooth 4.0,Bluetooth 5.5,ไม่มีข้อมูลนี้ในฐานข้อมูล,คำถามนี้ไม่เกี่ยวข้องกับฐานข้อมูลสินค้า
3,4,อยากเอามือถือเครื่องเก่ามาเทิร์น เปลี่ยนเป็นเค...,ได้ ลดราคาสูงสุด 30%,ได้ เฉพาะสมาร์ทโฟนสายฟ้าเท่านั้น,ได้ ต้องเป็นสมาชิก Gold ขึ้นไป,ได้ เฉพาะสินค้าที่ซื้อจากฟ้าใหม่,"ได้ แต่ต้องซื้อเครื่องใหม่ราคาเกิน 10,000 บาท",ไม่มีบริการ Trade-in ในขณะนี้,ได้ เฉพาะช่วง Mega Sale,ได้ ต้องนำไปที่สาขาเท่านั้น,ไม่มีข้อมูลนี้ในฐานข้อมูล,คำถามนี้ไม่เกี่ยวข้องกับฐานข้อมูลสินค้า
4,5,สั่งของจากฟ้าใหม่ สั่งได้ครั้งละกี่ชิ้นครับ,ไม่จำกัดจำนวน,5 รายการ,20 รายการ,15 รายการ,3 รายการ,10 รายการ,50 รายการ,8 รายการ,ไม่มีข้อมูลนี้ในฐานข้อมูล,คำถามนี้ไม่เกี่ยวข้องกับฐานข้อมูลสินค้า


In [3]:
# ======== Prompt template ========
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, use_fast=True, fix_mistral_regex=True)

# ======== RAG setup ========
embeddings = HuggingFaceEmbeddings(model_name=f"./models/{RETRIEVER}")
reranker = FlagReranker('./models/bge-reranker') 

vectorstore = FAISS.load_local(
    f"./vectorstore/{RETRIEVER}", 
    embeddings,
    allow_dangerous_deserialization=True
)
# gpu_index = faiss.index_cpu_to_gpu(faiss.StandardGpuResources(), 0, vectorstore.index)
# vectorstore.index = gpu_index

[2026-03-29 06:42:47] INFO SentenceTransformer.py:219: Use pytorch device_name: cuda:0
[2026-03-29 06:42:47] INFO SentenceTransformer.py:227: Load pretrained SentenceTransformer: ./models/bge-m3


## Data Prep

In [4]:
template = '''### Role
คุณคือระบบตอบคำถามอัตโนมัติ (QA System) ที่เชี่ยวชาญด้านข้อมูลสินค้า (Products), ข้อมูลร้านค้า (Store Info) และนโยบายต่างๆ (Policies) ของร้าน **ฟ้าใหม่ (FahMai)** หน้าที่ของคุณคือตอบคำถามแบบตัวเลือก (Multiple Choice) โดยอ้างอิงจาก "ข้อมูลอ้างอิง" ที่กำหนดให้เท่านั้น และตอบกลับเป็นรูปแบบ JSON เสมอ

### Context (ข้อมูลอ้างอิง)
{retrieved_context}

### Question (คำถาม)
{question}

### Options (ตัวเลือก)
1) {choice_1}
2) {choice_2}
3) {choice_3}
4) {choice_4}
5) {choice_5}
6) {choice_6}
7) {choice_7}
8) {choice_8}
9) {choice_9}
10) {choice_10}

### Instructions
1. วิเคราะห์คำถามและตรวจสอบตัวเลือกทั้งหมดเปรียบเทียบกับ "ข้อมูลอ้างอิง" ของร้านฟ้าใหม่
2. หากพบคำตอบที่ถูกต้องในข้อมูลอ้างอิง ให้ตอบเป็น **ตัวเลขลำดับข้อ (1-8)** ที่ถูกต้องที่สุดเพียงข้อเดียว
3. หากในข้อมูลอ้างอิงระบุชัดเจนว่า **ไม่มีข้อมูล** ดังกล่าว หรือข้อมูลไม่เพียงพอที่จะตอบ ให้ตอบข้อ **9**
4. หากคำถาม **ไม่เกี่ยวข้อง** กับสินค้า ข้อมูลร้าน หรือนโยบายของร้านฟ้าใหม่เลย ให้ตอบข้อ **10**
5. **ข้อห้ามสำคัญ:** ห้ามใช้ความรู้ภายนอกเด็ดขาด ห้ามเดาคำตอบ และห้ามตอบเป็นข้อความอื่นนอกเหนือจาก JSON format ที่กำหนด

### Output Format
{{"Answer": [หมายเลขข้อที่ถูกต้อง]}}
'''

In [5]:
def build_context_list(df, k_initial=15, k_final=3):
    context_list = []
    
    for idx, row in tqdm(df.iterrows(), total=len(df)):
        question_text = row['question']
        
        # 1. Initial Retrieval (Broad search)
        initial_docs = vectorstore.similarity_search(question_text, k=k_initial)
        
        # 2. Prepare pairs for Reranker: [[question, chunk1], [question, chunk2], ...]
        pairs = [[question_text, doc.page_content] for doc in initial_docs]
        
        # 3. Compute Scores
        scores = reranker.compute_score(pairs)
        
        # 4. Sort documents by score (highest to lowest)
        doc_score_pairs = sorted(zip(initial_docs, scores), key=lambda x: x[1], reverse=True)
        top_docs = [doc for doc, score in doc_score_pairs[:k_final]]
        
        # 5. Combine the top re-ranked chunks
        combined_context = "\n---\n".join([doc.page_content for doc in top_docs])
        context_list.append(combined_context)
        
    return context_list

context_list = build_context_list(test_df)
context_list[0]

100%|██████████| 100/100 [00:56<00:00,  1.76it/s]


'## รายละเอียดสินค้า\n\nเมื่อกีฬาและเทคโนโลยีมาบรรจบกัน วงโคจร Watch S3 Ultra คือจุดสูงสุดของสายฟิตเนสและการดำน้ำในรูปแบบนาฬิกาสมาร์ทวอทช์ ตัวเรือนผลิตจากไทเทเนียมเกรดการบินอวกาศ (Grade 5 Titanium) น้ำหนักเบากว่าสแตนเลสทั่วไปถึง 40% แต่แข็งแกร่งกว่าหลายเท่า ให้ความรู้สึกสวมใส่ที่เบาสบายตลอดทั้งวันแม้ระหว่างออกกำลังกายหนัก\n\nความโดดเด่นที่สุดของ S3 Ultra คือมาตรฐานกันน้ำระดับ 100 เมตร (10 ATM) พร้อมโหมด Dive Mode ที่รองรับการดำน้ำได้ถึง 40 เมตร บันทึกความลึก ความดัน อุณหภูมิน้ำ และเวลาดำน้ำแบบ real-time แสดงผลบนหน้าจอ AMOLED 1.9 นิ้วที่ยังคงอ่านได้ชัดเจนใต้น้ำ รองรับการดำน้ำทั้งแบบ recreational และ freediving\n\nระบบ GPS แบบ Dual-Band (L1+L5) ช่วยให้ติดตามเส้นทางได้แม่นยำกว่า GPS แบบ single-band ถึง 3 เท่า โดยเฉพาะในพื้นที่ที่มีตึกสูงหรือป่าทึบ รองรับดาวเทียม GPS, GLONASS, BeiDou, Galileo พร้อมกัน เหมาะสำหรับทั้งนักวิ่ง นักปั่น นักเดินทาง และนักกีฬากลางแจ้งระดับจริงจัง\n\nเซ็นเซอร์สุขภาพครบครัน ได้แก่ วัดชีพจร (HR) ตลอด 24 ชั่วโมง, ออกซิเจนในเลือด (SpO2), ECG (คลื่นไฟฟ้าหัวใจ), อุณหภูม

In [6]:
prompts = []
for i, row in test_df.iterrows():
    choices = {f"choice_{j}": str(row[f'choice_{j}']) for j in range(1, 11)}
    
    # Format the specific prompt for this row
    prompt = template.format(
        retrieved_context=context_list[i],
        question=row['question'],
        **choices
    )
    chat = [{"role": "user", "content": prompt}]
    chat = tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)
    prompts.append(chat)

print(prompts[0])

<|im_start|>user
### Role
คุณคือระบบตอบคำถามอัตโนมัติ (QA System) ที่เชี่ยวชาญด้านข้อมูลสินค้า (Products), ข้อมูลร้านค้า (Store Info) และนโยบายต่างๆ (Policies) ของร้าน **ฟ้าใหม่ (FahMai)** หน้าที่ของคุณคือตอบคำถามแบบตัวเลือก (Multiple Choice) โดยอ้างอิงจาก "ข้อมูลอ้างอิง" ที่กำหนดให้เท่านั้น และตอบกลับเป็นรูปแบบ JSON เสมอ

### Context (ข้อมูลอ้างอิง)
## รายละเอียดสินค้า

เมื่อกีฬาและเทคโนโลยีมาบรรจบกัน วงโคจร Watch S3 Ultra คือจุดสูงสุดของสายฟิตเนสและการดำน้ำในรูปแบบนาฬิกาสมาร์ทวอทช์ ตัวเรือนผลิตจากไทเทเนียมเกรดการบินอวกาศ (Grade 5 Titanium) น้ำหนักเบากว่าสแตนเลสทั่วไปถึง 40% แต่แข็งแกร่งกว่าหลายเท่า ให้ความรู้สึกสวมใส่ที่เบาสบายตลอดทั้งวันแม้ระหว่างออกกำลังกายหนัก

ความโดดเด่นที่สุดของ S3 Ultra คือมาตรฐานกันน้ำระดับ 100 เมตร (10 ATM) พร้อมโหมด Dive Mode ที่รองรับการดำน้ำได้ถึง 40 เมตร บันทึกความลึก ความดัน อุณหภูมิน้ำ และเวลาดำน้ำแบบ real-time แสดงผลบนหน้าจอ AMOLED 1.9 นิ้วที่ยังคงอ่านได้ชัดเจนใต้น้ำ รองรับการดำน้ำทั้งแบบ recreational และ freediving

ระบบ GPS แบบ Dual-Band (L1+L5) ช่ว

## Inference

In [7]:
model = LLM(
    model=MODEL_PATH,
    max_model_len=16000,
    tensor_parallel_size=1,
    enable_prefix_caching=True,
    gpu_memory_utilization=0.8,
    enforce_eager=True,
)

INFO 03-29 06:44:53 [utils.py:238] non-default args: {'max_model_len': 16000, 'enable_prefix_caching': True, 'gpu_memory_utilization': 0.8, 'disable_log_stats': True, 'enforce_eager': True, 'model': './models/openthaigpt-8b'}
INFO 03-29 06:44:53 [model.py:531] Resolved architecture: Qwen3ForCausalLM
INFO 03-29 06:44:53 [model.py:1554] Using max model len 16000
INFO 03-29 06:44:53 [scheduler.py:231] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 03-29 06:44:53 [vllm.py:747] Asynchronous scheduling is enabled.
WARNING 03-29 06:44:53 [vllm.py:781] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
WARNING 03-29 06:44:53 [vllm.py:792] Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ignored.
INFO 03-29 06:44:55 [vllm.py:957] Cudagraph is disabled under eager mode
WARNING 03-29 06:44:56 [system_utils.py:152] We must

Loading safetensors checkpoint shards:   0% Completed | 0/4 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  25% Completed | 1/4 [00:03<00:09,  3.27s/it]
Loading safetensors checkpoint shards:  50% Completed | 2/4 [00:05<00:05,  2.74s/it]
Loading safetensors checkpoint shards:  75% Completed | 3/4 [00:08<00:02,  2.94s/it]
Loading safetensors checkpoint shards: 100% Completed | 4/4 [00:10<00:00,  2.48s/it]
Loading safetensors checkpoint shards: 100% Completed | 4/4 [00:10<00:00,  2.65s/it]
(EngineCore_DP0 pid=3817017) 


(EngineCore_DP0 pid=3817017) INFO 03-29 06:45:24 [default_loader.py:293] Loading weights took 10.64 seconds
(EngineCore_DP0 pid=3817017) INFO 03-29 06:45:25 [gpu_model_runner.py:4338] Model loading took 15.27 GiB memory and 12.598327 seconds
(EngineCore_DP0 pid=3817017) INFO 03-29 06:45:27 [gpu_worker.py:424] Available KV cache memory: 15.46 GiB
(EngineCore_DP0 pid=3817017) INFO 03-29 06:45:27 [kv_cache_utils.py:1314] GPU KV cache size: 112,592 tokens
(EngineCore_DP0 pid=3817017) INFO 03-29 06:45:27 [kv_cache_utils.py:1319] Maximum concurrency for 16,000 tokens per request: 7.04x
(EngineCore_DP0 pid=3817017) INFO 03-29 06:45:27 [core.py:282] init engine (profile, create kv cache, warmup model) took 1.85 seconds
(EngineCore_DP0 pid=3817017) INFO 03-29 06:45:27 [vllm.py:747] Asynchronous scheduling is enabled.
(EngineCore_DP0 pid=3817017) WARNING 03-29 06:45:27 [vllm.py:781] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagra

In [8]:
decoding_params = SamplingParams(
    temperature=0.0, top_p=1, top_k=-1,
    max_tokens=8192,
    skip_special_tokens=True,
    repetition_penalty=1.15,
    frequency_penalty=0.2,
)

results = model.generate(prompts, decoding_params)
response = [r.outputs[0].text for r in results]
response[0]

Rendering prompts:   0%|          | 0/100 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/100 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

'<think>\nคำถามคือ "Watch S3 Ultra กันน้ำได้กี่ ATM ครับ" จากข้อมูลอ้างอิง เราสามารถหาคำตอบได้จากส่วน "สเปคสินค้า" ในตาราง:\n\n| รายการ | รายละเอียด |\n|--------|------------|\n| ... |\n| กันน้ำ | 100 เมตร (10 ATM) + Dive Mode (สูงสุด 40 เมตร) |\n| ... |\n\nดังนั้น Watch S3 Ultra กันน้ำได้ 10 ATM ซึ่งตรงกับตัวเลือกที่ 5.\n</think>\n\n\n```json\n{"Answer": 5}\n```'

In [9]:
def extract_choice_index(model_output):
    """
    Extracts the first integer found in the model's 'Answer' field.
    Handles: ["5", "10 ATM"], 5, "5", "ข้อ 5", and JSON strings.
    """
    model_output = model_output.split("</think>")[-1]
    try:
        # 1. If input is a string (JSON), parse it first
        if isinstance(model_output, str):
            try:
                data = json.loads(model_output)
                val = data.get("Answer", "")
            except json.JSONDecodeError:
                val = model_output # Fallback to raw string if JSON fails
        else:
            val = model_output.get("Answer", "")

        # 2. If the value is a list (like your example), take the first element
        if isinstance(val, list) and len(val) > 0:
            target = str(val[0])
        else:
            target = str(val)

        # 3. Use Regex to find the first number in the string (handles "ข้อ 2")
        match = re.search(r'\d+', target)
        if match:
            ans = int(match.group())
            return ans if ans <= 10 else 10
            
    except Exception as e:
        print(f"Error parsing: {model_output} | Error: {e}")
        
    return None

answers = [ extract_choice_index(r) for r in response]
answers[:5]

[5, 7, 2, 6, 6]

In [10]:
submission = pd.read_csv("./data/sample_submission.csv")
submission.loc[:, 'answer'] = answers
submission.to_csv("openthai_big-chugus.csv", index=False)

In [11]:
best_sub = pd.read_csv("openthai_rerank.csv")

# Merge with indicator=True
diff = best_sub.merge(submission, how='outer', indicator=True)

# Filter for rows that are NOT in 'both'
changes = diff[diff['_merge'] == 'right_only']
print("Difference rows from best submission:", changes.shape[0])

Difference rows from best submission: 23
